# 5 · Solving — Dirichlet dofs & static condensation

:::{dropdown} ▶ How to run / view this notebook
:class: howto-run

| 💻 Local | 🌐 Static | ▶ JupyterLite | ☁️ Colab |
|:--:|:--:|:--:|:--:|
| ✅ | [✅](/) | [✅](/lite/notebooks/index.html?path=05-solving.ipynb) | [✅](https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/05-solving.ipynb) |

<sub>✅ runs here · ⌛ runs but slowly (rough time)</sub>
:::

:::{dropdown} 🎭 Story — earning the Beast's trust
:class: storytelling

*You can cast a weak form, conjure geometry and wield functions on the mesh. The last
craft before the Beast truly trusts you is to understand the **solve** itself — how its
boundary is pinned, and how the system is tamed down to size.*
:::

**Solving** a Poisson problem looks like one innocent line,
`a.mat.Inverse(fes.FreeDofs()) * f.vec`. We look inside it for the part that trips
everyone up: **Dirichlet boundary conditions**. We meet the **free dofs**, learn to
**lift inhomogeneous boundary data** into the solution, and **shrink the system** with
**static condensation**.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw

mesh = Mesh(unit_square.GenerateMesh(maxh=0.3))

## 1. Free dofs — which unknowns do we actually solve for?

The `dirichlet` flag marks boundary dofs as **essential**: their values are
*prescribed*, not solved for. `fes.FreeDofs()` is a bit-array, `True` exactly on
the **remaining** (free) dofs. `Inverse(freedofs=…)` inverts the matrix only there,
leaving the constrained dofs untouched.

In [ ]:
fes = H1(mesh, order=4, dirichlet="bottom|right|top|left")
u, v = fes.TnT()
free = fes.FreeDofs()
print(f"total dofs {fes.ndof},  free {sum(1 for d in free if d)},  "
      f"fixed (Dirichlet) {sum(1 for d in free if not d)}")

## 2. Inhomogeneous Dirichlet data — *lifting*

What if $u=g\neq 0$ on the boundary? We cannot just solve on the free dofs — the
fixed boundary values *feed into* the equations. The trick is **lifting**: set the
boundary values into `gfu` with `Set(g, BND)`, move their effect to the
right-hand side as a **residual** $r=f-A\,u_{\partial}$, and solve for the free
correction only:

In [ ]:
g = sin(4 * x) * y                                   # some boundary data
a = BilinearForm(grad(u) * grad(v) * dx).Assemble()
f = LinearForm(1 * v * dx).Assemble()

gfu = GridFunction(fes)
gfu.Set(g, BND)                                      # prescribe u = g on the boundary
r = f.vec - a.mat * gfu.vec                          # residual carries the boundary effect
gfu.vec.data += a.mat.Inverse(free, inverse="sparsecholesky") * r
Draw(gfu, mesh, "u with u=g on the boundary")

## 3. Static condensation — solve a smaller system

A high-order space has many **internal** dofs (bubbles living *inside* one element)
coupled only to that element. They can be **eliminated locally** before the global
solve — *static condensation* — leaving a much smaller system on the **interface
(coupling)** dofs. NGSolve does the bookkeeping if you pass `condense=True`; the
global solve then runs on `FreeDofs(coupling=True)`.

In [ ]:
ac = BilinearForm(grad(u) * grad(v) * dx, condense=True).Assemble()
n_couple = sum(1 for d in fes.FreeDofs(coupling=True) if d)
print(f"global solve shrinks from {sum(1 for d in free if d)} free dofs "
      f"to {n_couple} coupling dofs")

# the full inverse = harmonic extension ∘ (Schur solve) ∘ extension-transpose
#                    + the element-local inner solve
invS = ac.mat.Inverse(fes.FreeDofs(coupling=True), inverse="sparsecholesky")
ext = IdentityMatrix() + ac.harmonic_extension
extT = IdentityMatrix() + ac.harmonic_extension_trans
inv = ext @ invS @ extT + ac.inner_solve

gfc = GridFunction(fes)
gfc.Set(g, BND)
gfc.vec.data += ac.harmonic_extension * gfc.vec      # lift boundary data, condensed
gfc.vec.data += inv * (f.vec - ac.mat * gfc.vec)
print(f"condensed vs plain solve differ by {sqrt(Integrate((gfu - gfc)**2, mesh)):.1e} "
      f"(same answer, smaller system)")

:::{dropdown} 📚 Further reading
:class: further-reading

- **Dirichlet boundary conditions** — i-tutorial
  [1.3 Dirichlet](https://docu.ngsolve.org/latest/i-tutorials/unit-1.3-dirichlet/dirichlet.html).
- **Static condensation** — i-tutorial
  [1.4 Static condensation](https://docu.ngsolve.org/latest/i-tutorials/unit-1.4-staticcond/staticcond.html).
:::

:::{dropdown} 🧠 Quiz — when is static condensation worth it?
:class: quiz
Most when the space has **many internal dofs per element** — high polynomial order,
or **bubble/DG-like** spaces, especially in 3D where internal dofs dominate. The
coupling system can be several times smaller, so a (sparse) direct factorisation is
much cheaper, and iterative solvers see a better-conditioned operator. For lowest
order ($p=1$, no interior dofs) there is nothing to condense. The same `condense`
machinery underlies **hybrid** methods (notebook 6's solver toolbox goes further).
:::

**Next:** the solve above used a direct factorisation. Unit 6 — the close of Part I —
opens the **solver toolbox**: the iterative methods and preconditioners that scale to
problems a direct solver can no longer swallow.

In [ ]:
# Navigation to the next unit — shown only in a live notebook (Colab /
# JupyterLite / local Jupyter), never in the rendered website.
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _nb, _title = "06-linear-solvers", "6 · The solver toolbox 🛠"
    if "google.colab" in sys.modules:
        _u = "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
    else:                                           # JupyterLite & local open relative .ipynb links
        _u = _nb + ".ipynb"
    from IPython.display import display, Markdown
    display(Markdown("➡️ **Next unit:** [" + _title + "](" + _u + ")"))